# Leocherd Iris Energy - Kaggle Experiment 04: Temporal Motion Smoothing
Evaluates One Euro adaptive low-pass filter vs Savitzky-Golay on noisy trajectories.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

class OneEuroFilter:
    def __init__(self, freq=30.0, min_cutoff=1.0, beta=0.007):
        self.freq, self.min_cutoff, self.beta = freq, min_cutoff, beta
        self.x_prev, self.dx_prev, self.t_prev = None, 0.0, None

    def filter(self, x, t):
        if self.t_prev is None:
            self.x_prev, self.t_prev = x, t
            return x
        dt = max(1e-4, t - self.t_prev)
        self.t_prev = t
        dx = (x - self.x_prev) / dt
        cutoff = self.min_cutoff + self.beta * abs(dx)
        alpha = 1.0 / (1.0 + (1.0 / (2.0 * math.pi * cutoff)) / dt)
        hat_x = alpha * x + (1.0 - alpha) * self.x_prev
        self.x_prev = hat_x
        return hat_x

# Benchmark on noisy sine wave
times = np.linspace(0, 2.0, 60)
true_signal = np.sin(2 * np.pi * 1.5 * times)
noisy_signal = true_signal + np.random.normal(0, 0.15, size=len(times))

f = OneEuroFilter()
smoothed = [f.filter(val, t) for val, t in zip(noisy_signal, times)]
print(f"MSE Noisy: {np.mean((noisy_signal - true_signal)**2):.4f}")
print(f"MSE Smoothed: {np.mean((np.array(smoothed) - true_signal)**2):.4f}")
